In [31]:
import os
import torch
import gc
from dotenv import load_dotenv
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextStreamer
from huggingface_hub import login
import gradio as gr

In [ ]:
# open-source models

LLAMA_3_1_8B = "meta-llama/Llama-3.1-8B-Instruct"
LLAMA_3_2_3B = "meta-llama/Llama-3.2-3B-Instruct"
LLAMA_3_2_1B = "meta-llama/Llama-3.2-1B-Instruct"
GEMMA = "google/gemma-3-270m-it"
PHI = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [3]:
load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [15]:
messages = [
    {"role": "user", "content": "Create a dataset of 5 objects each being unique. Each object should have a first name, last name, and email. Provide in json format."}
]

In [5]:
# quantization will reduces memory usage at the cost of precision

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [16]:
# tokenize the inputs

tokenizer = AutoTokenizer.from_pretrained(LLAMA_3_2_3B)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [17]:
inputs

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,   2947,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,   4110,    264,  10550,    315,    220,     20,
           6302,   1855,   1694,   5016,     13,   9062,   1665,   1288,    617,
            264,   1176,    836,     11,   1566,    836,     11,    323,   2613,
             13,  40665,    304,   3024,   3645,     13, 128009]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA_3_2_3B, device_map="auto", quantization_config=quant_config)

W0326 15:04:15.265000 94660 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 1/254 [00:00<04:08,  1.02it/s]c:\Users\jayma\Desktop\projs\dataset-generator\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 254/254 [00:07<00:00, 32.14it/s] 


Memory footprint: 2,197.6 MB


In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

In [10]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [18]:
outputs = model.generate(**inputs, max_new_tokens=80)
outputs[0]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
c:\Users\jayma\Desktop\projs\dataset-generator\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   1627,   2947,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,   4110,    264,  10550,    315,    220,     20,
          6302,   1855,   1694,   5016,     13,   9062,   1665,   1288,    617,
           264,   1176,    836,     11,   1566,    836,     11,    323,   2613,
            13,  40665,    304,   3024,   3645,     13, 128009, 128006,  78191,
        128007,    271,   8586,    596,    264,   4823,  10550,    315,    220,
            20,   5016,   6302,     11,   1855,    449,    264,   1176,    836,
            11,   1566,    836,     11,    323,   2613,   1473,  14196,   4077,
          9837,    220,    341,    262,    330,  29087,    794,    330,  85148,
           761,    262,    330,  30256,    794,    330,  28205,    301,    761,
           262,    330,   2386,    794, 

In [19]:
tokenizer.decode(outputs[0])

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Mar 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nCreate a dataset of 5 objects each being unique. Each object should have a first name, last name, and email. Provide in json format.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere\'s a JSON dataset of 5 unique objects, each with a first name, last name, and email:\n\n```\n[\n  {\n    "firstName": "Emily",\n    "lastName": "Patel",\n    "email": "emilypatel@example.com"\n  },\n  {\n    "firstName": "Liam",\n    "lastName": "Davis",\n    "'

In [38]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

In [32]:
def generate(model, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=False).to("cuda")
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
    streamer = TextStreamer(tokenizer)
    
    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
    outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)
    

In [37]:
generate(GEMMA, messages, False, 600)

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 3821.00it/s]
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<bos><start_of_turn>user
Create a dataset of 5 objects each being unique. Each object should have a first name, last name, and email. Provide in json format.<end_of_turn>
<start_of_turn>model
```json
[
  {
    "object_id": 1,
    "first_name": "Alice",
    "last_name": "Smith",
    "email": "alice.smith@example.com"
  },
  {
    "object_id": 2,
    "first_name": "Bob",
    "last_name": "Johnson",
    "email": "bob.johnson@example.com"
  },
  {
    "object_id": 3,
    "first_name": "Charlie",
    "last_name": "Brown",
    "email": "charlie.brown@example.com"
  },
  {
    "object_id": 4,
    "first_name": "David",
    "last_name": "Lee",
    "email": "david.lee@example.com"
  },
  {
    "object_id": 5,
    "first_name": "Eve",
    "last_name": "Davis",
    "email": "eve.davis@example.com"
  }
]
```<end_of_turn>


In [ ]:
generator = pipeline("text-generation", device="cuda")
result = generator(messages)
print(result)